# Thursday AI — Fine-tune Qwen2.5-3B-Instruct on Kaggle T4×2

**Goal:** turn `Qwen/Qwen2.5-3B-Instruct` into **Thursday AI v0.1** — an offline, voice-first, autonomous OS-controlling assistant — via 4-bit QLoRA fine-tuning on the Thursday-AI dataset.

This notebook uses **Unsloth** (2× faster, 70 % less VRAM) on **Kaggle T4 ×2** (16 GB × 2 = 32 GB pooled VRAM). It loads the dataset directly from the GitHub repo, trains, merges the LoRA, converts to GGUF `Q4_K_M`, and pushes the final model back to HuggingFace Hub.

**Run order:**
1. Set Kaggle Secrets: `HF_TOKEN` (HuggingFace token, write access)
2. Set accelerator: `GPU T4 x2`
3. Run all cells top to bottom. Total wall-clock: ~6-8 hrs for 90k examples × 3 epochs.

---

**Repo:** https://github.com/skyro777/Thursday_AI  
**Living plan:** `LIVING_PLAN.md`  
**Dataset spec:** `research/02_training_data_approach.md`  
**Recipe reference:** `research/03_finetuning_recipe_kaggle.md`

## 0. Config

Edit these if you forked the repo or want a different model.

In [ ]:
CONFIG = {
    # --- Source ---
    'github_repo'       : 'https://github.com/skyro777/Thursday_AI.git',
    'github_branch'     : 'main',
    'repo_dir'          : '/kaggle/working/Thursday_AI',

    # --- Model ---
    'base_model'        : 'Qwen/Qwen2.5-3B-Instruct',
    'max_seq_length'    : 3072,   # M0-fix-24: was 4096 (would take ~20h — too long for Kaggle 12h limit). 3072 fits ~95% of examples; only longest get truncated.
    'load_in_4bit'      : True,
    'dtype'             : None,  # None = auto (T4 -> fp16)

    # --- LoRA ---
    'lora_r'            : 32,
    'lora_alpha'        : 64,
    'lora_dropout'      : 0,   # must be 0 to enable Unsloth's fast fused LoRA kernels
    'lora_target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                            'gate_proj', 'up_proj', 'down_proj'],

    # --- Training ---
    'per_device_batch'  : 1,   # M0-fix-24: keep at 1 (seq=3072 + bs=2 would risk OOM on T4)
    'grad_accum_steps'  : 8,   # M0-fix-24: 8 instead of 16 — fewer micro-batches per step = faster. Effective batch = 8 (acceptable for 5k examples).
    'epochs'            : 1,   # M0-fix-24: was 3. For smoke test, 1 epoch is enough to verify learning. Real v0.1 will use 3 epochs on 90k examples.
    # --- Dataset size ---
    # smoke_mode=True: 5k examples, ~1 min training (plumbing test only)
    # smoke_mode=False: 90k examples, ~6-8 hours (real v0.1 training)
    'smoke_mode'       : True,   # M0-fix-20: turn off for real training
    'smoke_dataset_size'  : 2000,    # M0-fix-24: reduced from 5000 to 2000 — fits in ~2 hours, leaves headroom for merge+GGUF+upload
    'full_dataset_size'   : 90000,   # target for Phase 2 (requires running generate_llm_distilled.py first)
    'learning_rate'     : 2e-4,
    'warmup_ratio'      : 0.03,
    'lr_scheduler'      : 'cosine',
    'weight_decay'      : 0.01,
    'optim'             : 'paged_adamw_8bit',
    'save_steps'        : 500,
    'logging_steps'     : 10,
    'seed'              : 42,

    # --- Output ---
    'output_dir'        : '/kaggle/working/thursday-ai-lora',
    'merged_dir'        : '/kaggle/working/thursday-ai-merged',
    'gguf_path'         : '/kaggle/working/thursday-ai-v0.1-Q4_K_M.gguf',
    'gguf_quant'        : 'Q4_K_M',

    # --- HF Hub upload (optional) ---
    # M0-fix-21: namespace will be filled in cell 2 from whoami() — don't hardcode
    # (your HF username may differ from your GitHub username!)
    'hf_repo_merged'    : 'thursday-ai-v0.1-merged',   # becomes <username>/thursday-ai-v0.1-merged
    'hf_repo_gguf'      : 'thursday-ai-v0.1-gguf',       # becomes <username>/thursday-ai-v0.1-gguf
}

# Print effective batch size so we can sanity-check before training
import torch
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
eff = CONFIG['per_device_batch'] * CONFIG['grad_accum_steps']  # single-GPU
print(f'GPUs detected: {n_gpus} (using 1 for training — Unsloth recommends 1 GPU on Kaggle T4x2 due to no NVLink)')
print(f'Effective batch size: {eff}')
assert n_gpus >= 1, f'Expected at least 1 GPU, got {n_gpus}.'


## 1. Install dependencies

Unsloth is the main accelerator. We also pin `transformers`, `peft`, `trl`, `bitsandbytes`, `accelerate`, and `huggingface_hub` to versions known to work together on Kaggle T4.

In [ ]:
# M0-fix-17: set CUDA allocator env vars BEFORE any imports.
# expandable_segments:True reduces fragmentation (the OOM error itself recommends this).
# Must be set before torch.cuda is initialised.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF']               = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM']                = 'false'   # silence warning
os.environ['TRANSFORMERS_ATTENTION_IMPLEMENTATION'] = 'sdpa'    # faster + less memory on T4

# ====================================================================
# THURSDAY AI — FINAL KAGGLE INSTALL (research-verified by inspecting 7 wheels)
# See: research/04_kaggle_unsloth_working_config.md
#
# Strategy: Kaggle preinstalls a working stack (torch 2.10, transformers 5.0,
# hub 1.11, datasets 5.0, etc.). We only override what's broken:
#   - TRL (Kaggle's is newer than 0.12 — incompatible API with our notebook)
#   - bitsandbytes (Kaggle's is too old for transformers 5.0's 4-bit quant requirement)
# Plus install Unsloth + unsloth_zoo with --no-deps so they don't disturb Kaggle's stack.
#
# Verified compatibility chain (by downloading wheels from PyPI):
#   - unsloth 2026.9.14 requires unsloth_zoo>=2026.9.9 ✓ (we pin both)
#   - bitsandbytes 0.49.2 requires torch<3,>=2.3 ✓ (Kaggle has 2.10)
#   - bitsandbytes 0.49.2 has libbitsandbytes_cuda128.so ✓ (Kaggle is CUDA 12.8)
#   - bitsandbytes 0.49.2 satisfies transformers 5.0's >=0.46.1 requirement ✓
#   - bitsandbytes 0.49.2 < 0.50 ✓ (0.50+ requires torch 2.11, Kaggle has 2.10)
#   - TRL 0.12.1 has SFTConfig + SFTTrainer ✓ (matches our cell 7 SFTTrainer API)
#   - transformers 5.0 is technically excluded by Unsloth's metadata but
#     Unsloth collaborator @Datta0 confirmed --no-deps bypass works
#     (GitHub issue #4022). Verified by user's M0-fix-13 worklog.
# ====================================================================

!pip install -q --upgrade pip

# 1. Install Unsloth + unsloth_zoo (pinned for reproducibility) with --no-deps.
!pip install -q --upgrade --force-reinstall --no-cache-dir --no-deps \
    "unsloth==2026.9.14" "unsloth_zoo==2026.9.9"

# 2. Override TRL to 0.12.1 (last version with our cell 7 SFTTrainer API).
!pip install -q --upgrade --force-reinstall --no-deps "trl==0.12.1"

# 3. Install bitsandbytes 0.49.2 — verified by wheel inspection.
#    If 0.49.2 fails to download, fall back to 0.48.2 (also has cuda128 binary).
!pip install -q --upgrade --force-reinstall --no-deps "bitsandbytes==0.49.2" || \
    !pip install -q --upgrade --force-reinstall --no-deps "bitsandbytes==0.48.2"

# 4. GGUF conversion tools (used later in the notebook).
!pip install -q "gguf>=0.6.0"
!apt-get -y install -q git-lfs

# ====================================================================
# SANITY CHECK — imports + version prints.
# CRITICAL: import unsloth FIRST. It patches trl/transformers/peft on import.
# If they're already loaded, the patches don't apply (UserWarning).
# We do NOT do clever bitsandbytes introspection — that broke 4 prior attempts.
# If 4-bit quant is broken, the model load cell will tell us with a real error.
# ====================================================================
import unsloth  # MUST be first
import torch, trl, transformers, peft, accelerate, bitsandbytes, huggingface_hub, tokenizers, datasets, gguf
print(f'torch           {torch.__version__}')
print(f'transformers    {transformers.__version__}')
print(f'tokenizers      {tokenizers.__version__}')
print(f'trl             {trl.__version__}')
print(f'peft            {peft.__version__}')
print(f'accelerate      {accelerate.__version__}')
print(f'bitsandbytes    {bitsandbytes.__version__}')
print(f'huggingface_hub {huggingface_hub.__version__}')
print(f'datasets        {datasets.__version__}')
print(f'GPU count: {torch.cuda.device_count()}')
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, PeftModel
from huggingface_hub import HfApi, login
from datasets import load_dataset, Dataset
print('OK: all imports succeeded. Proceed to cell 2.')

## 2. Authenticate with HuggingFace

Add a Kaggle Secret named `HF_TOKEN` with a HuggingFace access token (write permission) before running this cell.

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN to Kaggle Secrets'

# M0-fix: we set add_to_git_credential=False because Kaggle notebooks don't
# ship a git credential helper, and we don't need one.
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to HuggingFace Hub. Token is in-memory for this session.')

# M0-fix-20: VERIFY HF TOKEN PERMISSIONS IMMEDIATELY.
# Previously the token was only checked at the very end during upload,
# wasting hours if the token lacked write permission. Now we check
# at the start so we know in 5 seconds, not 5 hours.
from huggingface_hub import HfApi, whoami
import sys
try:
    user_info = whoami(token=HF_TOKEN)
    username = user_info['name']
    print(f'Token is valid. Logged in as: {username}')

    # M0-fix-21: Use the ACTUAL HF username (not hardcoded 'skyro777').
    # Your HF namespace may differ from your GitHub username.
    repo_base_merged = CONFIG['hf_repo_merged'].split('/')[-1]
    repo_base_gguf = CONFIG['hf_repo_gguf'].split('/')[-1]
    CONFIG['hf_repo_merged'] = f'{username}/{repo_base_merged}'
    CONFIG['hf_repo_gguf']   = f'{username}/{repo_base_gguf}'
    print(f'Will upload merged model to: {CONFIG["hf_repo_merged"]}')
    print(f'Will upload GGUF to:          {CONFIG["hf_repo_gguf"]}')

    # Check that the token can write to user namespace
    api = HfApi(token=HF_TOKEN)
    try:
        token_info = api.whoami()
        auth_info = token_info.get('auth', {})
        accesses = auth_info.get('access', {})
        if isinstance(accesses, dict):
            scoped = accesses.get('scoped', [])
        else:
            scoped = accesses if isinstance(accesses, list) else []

        has_write = any(
            ('repo.write' in str(s).lower()) or
            ('content.write' in str(s).lower()) or
            ('write' in str(s).lower() and 'repo' in str(s).lower())
            for s in scoped
        )

        if has_write:
            print('Token has WRITE permission')
        else:
            print(f'WARNING: token may not have write permission. Scopes: {scoped}')
            print('To fix: create a new token at https://huggingface.co/settings/tokens')
            print('        with Write or Full Access preset, then update the Kaggle Secret.')
    except Exception as scope_err:
        print(f'Token is valid (could not verify write scope automatically: {scope_err})')
        print('If upload fails at the end, regenerate token with Write or Full Access.')

except Exception as e:
    print(f'ERROR: HF_TOKEN validation failed: {e}', file=sys.stderr)
    print('', file=sys.stderr)
    print('Possible causes:', file=sys.stderr)
    print('  1. Kaggle Secret HF_TOKEN not set, has a typo in name, or is empty', file=sys.stderr)
    print('  2. Token was not copied correctly (truncated, has whitespace)', file=sys.stderr)
    print('  3. Token has been revoked or expired', file=sys.stderr)
    print('', file=sys.stderr)
    print('Fix: go to https://huggingface.co/settings/tokens', file=sys.stderr)
    print('     Create a new token with Write or Full Access preset', file=sys.stderr)
    print('     Copy it exactly (no whitespace)', file=sys.stderr)
    print('     In Kaggle: Add-ons -> Secrets -> Add Secret', file=sys.stderr)
    print('                Label: HF_TOKEN  (case-sensitive, exactly this)', file=sys.stderr)
    print('                Value: <paste token>', file=sys.stderr)
    print('     Restart the kernel and re-run this cell.', file=sys.stderr)
    raise

## 3. Clone the Thursday AI repo and load the dataset

The repo contains `data/examples/*.jsonl` (hand-written golden examples) and the output of `scripts/generate_template_data.py` (if you ran it locally and committed the result). 

If the template-generated file isn't committed yet, we generate it here from the templates + examples folder.

In [ ]:
import os, subprocess, json
from pathlib import Path

if not Path(CONFIG['repo_dir']).exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        '-b', CONFIG['github_branch'],
        CONFIG['github_repo'], CONFIG['repo_dir'],
    ], check=True)
else:
    print('Repo already cloned.')

REPO = Path(CONFIG['repo_dir'])

# If the template-generated file is committed, use it; otherwise generate it on the fly
processed_dir = REPO / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
dataset_path = processed_dir / 'template_generated.jsonl'

if not dataset_path.exists() or dataset_path.stat().st_size < 1000:
    print('Template-generated dataset not found in repo — generating 5,000 examples...')
    subprocess.run([
        'python', str(REPO / 'scripts' / 'generate_template_data.py'),
        '--count', str(CONFIG['smoke_dataset_size'] if CONFIG['smoke_mode'] else CONFIG['full_dataset_size']),
        '--seed', str(CONFIG['seed']),
        '--out', str(dataset_path),
    ], check=True)

# Validate
subprocess.run([
    'python', str(REPO / 'scripts' / 'validate_dataset.py'),
    str(dataset_path),
    str(REPO / 'data' / 'examples'),
    '--max-errors', '50',
], check=False)

# Concatenate all JSONL sources into one master file
master_path = processed_dir / 'master.jsonl'
with master_path.open('w') as out:
    for jf in sorted((REPO / 'data' / 'examples').glob('*.jsonl')):
        out.write(jf.read_text())
    out.write(dataset_path.read_text())

n_examples = sum(1 for _ in master_path.open())
print(f'Master dataset: {master_path} — {n_examples} examples')

## 4. Load the base model with Unsloth (4-bit QLoRA)

Loads `Qwen/Qwen2.5-3B-Instruct` in 4-bit NF4 quantization on GPU 0. We'll DDP across both T4s via `accelerate`.

In [ ]:
# ====================================================================
# Load Qwen2.5-3B-Instruct in 4-bit NF4 via Unsloth.
# Unsloth's FastLanguageModel is ~2x faster than vanilla bitsandbytes load
# and patches the model with fused LoRA kernels.
# ====================================================================
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = CONFIG['base_model'],
    max_seq_length = CONFIG['max_seq_length'],
    dtype = CONFIG['dtype'],  # None = auto (fp16 on T4)
    load_in_4bit = CONFIG['load_in_4bit'],
    # M0-fix-17: NO device_map — load on GPU 0 only.
    # device_map='auto' was forcing model-parallel sharding across both T4s
    # (single process), which concentrated the LM head + logits on GPU 1
    # and caused OOM. Unsloth explicitly recommends single-GPU on Kaggle T4x2:
    # https://huggingface.co/unsloth/Qwen2.5-3B-Instruct
    # 'Kaggle has 2x T4s, but we use 1. Due to overhead, 1x T4 is 5x faster.'
)
print(f'Loaded {CONFIG["base_model"]} in 4-bit. Vocab size: {len(tokenizer)}')

## 5. Attach LoRA adapters

All-linear-target LoRA with `r=32, alpha=64`. This is the standard agentic-LoRA recipe and works well for tool-use fine-tuning.

In [ ]:
# ====================================================================
# Attach LoRA adapters via Unsloth's FastLanguageModel.get_peft_model.
# This patches the model with fused LoRA kernels for ~2x training speedup.
# lora_dropout MUST be 0 to enable the fast fused path.
# ====================================================================
model = FastLanguageModel.get_peft_model(
    model,
    r = CONFIG['lora_r'],
    target_modules = CONFIG['lora_target_modules'],
    lora_alpha = CONFIG['lora_alpha'],
    lora_dropout = CONFIG['lora_dropout'],  # 0 — enables fused kernels
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = CONFIG['seed'],
    use_rslora = False,
    loftq_config = None,
)
print('LoRA adapters attached via Unsloth.')
model.print_trainable_parameters()

## 6. Format dataset with the Qwen2 tool-call chat template

Each example has `{system, tools, messages}`. We apply Qwen2's native chat template (which renders the `tools` array into the `<tools>` block and `tool_calls` into the right assistant-turn format), then mask everything except assistant responses with `train_on_responses_only`.

In [ ]:
from datasets import load_dataset, Dataset

raw = load_dataset('json', data_files=str(master_path), split='train')
print(f'Loaded {len(raw)} examples from {master_path}')

def format_example(ex):
    """Apply Qwen2 chat template to a {system, tools, messages} example."""
    # Qwen2 chat template expects messages with system, user, assistant, tool roles.
    # It renders `tools` into a <tools>...</tools> block automatically.
    messages = []
    if ex.get('system'):
        messages.append({'role': 'system', 'content': ex['system']})
    messages.extend(ex['messages'])
    text = tokenizer.apply_chat_template(
        messages,
        tools=ex['tools'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

formatted = raw.map(format_example, remove_columns=raw.column_names)
print(f'Formatted {len(formatted)} examples. Sample:')
print(formatted[0]['text'][:600] + '...')
# M0-fix-23: Check token length distribution — if examples exceed max_seq_length, they get truncated and lost!
token_lengths = [len(tokenizer(ex['text'])['input_ids']) for ex in formatted.select(range(min(200, len(formatted))))]
import numpy as np
tl = np.array(token_lengths)
print(f'Token length distribution (first {len(tl)} examples):')
print(f'  min={tl.min()}, max={tl.max()}, mean={tl.mean():.0f}, median={np.median(tl):.0f}')
print(f'  examples > {CONFIG["max_seq_length"]} tokens: {(tl > CONFIG["max_seq_length"]).sum()} / {len(tl)} ({100*(tl > CONFIG["max_seq_length"]).sum()/len(tl):.1f}%)')
if (tl > CONFIG['max_seq_length']).sum() > len(tl) * 0.1:
    print(f'WARNING: more than 10% of examples exceed max_seq_length={CONFIG["max_seq_length"]}!')
    print(f'These will be truncated and their assistant responses may be cut off → loss masking fails → example removed from training.')
    print(f'Fix: increase max_seq_length in CONFIG, or shorten the system prompt / tools list.')
else:
    print(f'OK: examples fit within max_seq_length={CONFIG["max_seq_length"]}')


## 7. Set up the SFTTrainer with response-only loss masking

`train_on_responses_only` ensures we compute loss ONLY on assistant turns (including the tool_call JSON), not on system / user / tool messages. This is critical — otherwise the model wastes capacity learning to copy user utterances.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model = model,
    # M0-fix-15: swapped `tokenizer=` -> `processing_class=` for forward-compat with TRL >= 0.16.
    # With TRL 0.12.1, both names work via Unsloth's auto-conversion shim,
    # but `processing_class` is the future-proof name.
    processing_class = tokenizer,
    train_dataset = formatted,
    dataset_text_field = 'text',
    max_seq_length = CONFIG['max_seq_length'],
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding='longest'),
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = CONFIG['per_device_batch'],
        gradient_accumulation_steps = CONFIG['grad_accum_steps'],
        warmup_ratio = CONFIG['warmup_ratio'],
        num_train_epochs = CONFIG['epochs'],
        learning_rate = CONFIG['learning_rate'],
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = CONFIG['logging_steps'],
        optim = CONFIG['optim'],
        weight_decay = CONFIG['weight_decay'],
        lr_scheduler_type = CONFIG['lr_scheduler'],
        seed = CONFIG['seed'],
        output_dir = CONFIG['output_dir'],
        save_steps = CONFIG['save_steps'],
        save_total_limit = 3,
        report_to = 'none',
    ),
)

# Mask loss on everything except assistant responses.
# Unsloth occasionally moves this helper around — try a few import paths.
train_on_responses_only = None
for _import_path in ('unsloth.trainers', 'unsloth', 'unsloth.models'):
    try:
        _mod = __import__(_import_path, fromlist=['train_on_responses_only'])
        train_on_responses_only = getattr(_mod, 'train_on_responses_only', None)
        if train_on_responses_only is not None:
            print(f'Loaded train_on_responses_only from {_import_path}')
            break
    except Exception:
        continue

if train_on_responses_only is not None:
    try:
        trainer = train_on_responses_only(
            trainer,
            instruction_part = '<|im_start|>user\n',
            response_part = '<|im_start|>assistant\n',
        )
        print('Loss masking applied: only assistant turns contribute to loss.')
    except Exception as e:
        print(f'Warning: could not apply train_on_responses_only ({e}). Training will compute loss on all tokens.')
else:
    print('Warning: train_on_responses_only not found. Training without explicit loss masking.')

print('Trainer ready.')

## 8. Train!

On T4 ×2 with 5,000 examples × 3 epochs at effective batch 16, this is ~1,000 steps. Should take ~30-45 minutes.

For the full 90k dataset it'll be ~17,000 steps and ~6-8 hours (within Kaggle's 12-hr session limit).

In [ ]:
import time
t0 = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - t0
print(f'\nTraining done in {elapsed/60:.1f} minutes.')
print(f'Final loss: {trainer_stats.training_loss:.4f}')

## 9. Save the LoRA adapter

We save just the adapter (~80 MB) so we can resume training or apply it to the base later. We also save the full merged model in the next step.

In [ ]:
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f'LoRA adapter saved to {CONFIG["output_dir"]}')
!ls -lh {CONFIG['output_dir']}

## 10. Merge LoRA into base + convert to GGUF Q4_K_M

Now we merge the adapter weights into the base model (so we don't need the LoRA at inference time) and convert the merged model to a single GGUF file quantized to Q4_K_M (~1.93 GB).

We use the repo's `scripts/merge_and_quantize.py` which handles the merge via PEFT and conversion via llama.cpp's `convert_hf_to_gguf.py`.

In [ ]:
# Merge LoRA into base
import subprocess, os
os.environ['HF_TOKEN'] = HF_TOKEN

# Merge
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print('Loading base for merge...')
base = AutoModelForCausalLM.from_pretrained(
    CONFIG['base_model'],
    torch_dtype=torch.float16,
    device_map='auto',
)
print('Attaching adapter...')
merged = PeftModel.from_pretrained(base, CONFIG['output_dir'])
merged = merged.merge_and_unload()

Path(CONFIG['merged_dir']).mkdir(parents=True, exist_ok=True)
print(f'Saving merged model to {CONFIG["merged_dir"]}...')
merged.save_pretrained(CONFIG['merged_dir'], safe_serialization=True)
tok = AutoTokenizer.from_pretrained(CONFIG['base_model'])
tok.save_pretrained(CONFIG['merged_dir'])
print('Merged model saved.')

In [ ]:
# ====================================================================
# Build llama.cpp and convert the merged model to GGUF Q4_K_M.
#
# M0-fix-19: llama.cpp REMOVED its Makefile build system — now CMake only.
# Error was: 'Makefile:6: *** Build system changed: The Makefile build has
# been replaced by CMake. For build instructions see:
# https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md'
#
# New build process (per llama.cpp docs):
#   cmake -B build
#   cmake --build build --config Release -j 4
# Then the binary is at build/bin/llama-quantize (not the repo root).
# ====================================================================
import subprocess, os
from pathlib import Path
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['LLAMA_CPP_PATH'] = CONFIG.get('llama_cpp_dir', '/kaggle/working/llama.cpp')

llama_cpp_dir = os.environ['LLAMA_CPP_PATH']
if not Path(llama_cpp_dir).exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggml-org/llama.cpp', llama_cpp_dir], check=True)

# Build with CMake (Makefile was removed in late 2025)
build_dir = Path(llama_cpp_dir) / 'build'
if not (build_dir / 'bin' / 'llama-quantize').exists():
    print('Configuring CMake build...')
    subprocess.run(['cmake', '-B', str(build_dir), '-S', llama_cpp_dir,
                    '-DCMAKE_BUILD_TYPE=Release'], check=True)
    print('Building llama-quantize (this takes ~2-3 min)...')
    subprocess.run(['cmake', '--build', str(build_dir), '--config', 'Release',
                    '--target', 'llama-quantize', '-j', '4'], check=True)

quantize_bin = str(build_dir / 'bin' / 'llama-quantize')
assert Path(quantize_bin).exists(), f'llama-quantize not built at {quantize_bin}'
print(f'llama-quantize binary: {quantize_bin}')

# Convert merged HF -> f16 GGUF using the convert script
f16_gguf = CONFIG['gguf_path'].replace('.gguf', '.f16.gguf')
subprocess.run([
    'python', f'{llama_cpp_dir}/convert_hf_to_gguf.py',
    CONFIG['merged_dir'],
    '--outtype', 'f16',
    '--outfile', f16_gguf,
], check=True)
print(f'f16 GGUF written: {f16_gguf}')

# Quantize f16 -> Q4_K_M
subprocess.run([
    quantize_bin,
    f16_gguf, CONFIG['gguf_path'], CONFIG['gguf_quant'],
], check=True)
print(f'Q4_K_M GGUF written: {CONFIG["gguf_path"]}')

size_mb = Path(CONFIG['gguf_path']).stat().st_size / (1024*1024)
print(f'Size: {size_mb:.1f} MB')

## 11. Push to HuggingFace Hub

Uploads the merged model and the GGUF to HF Hub so they're downloadable by `ollama` or `llama.cpp` from anywhere.

In [ ]:
from huggingface_hub import HfApi, create_repo
api = HfApi(token=HF_TOKEN)

# Upload merged HF model
create_repo(CONFIG['hf_repo_merged'], exist_ok=True, token=HF_TOKEN)
api.upload_folder(folder_path=CONFIG['merged_dir'], repo_id=CONFIG['hf_repo_merged'], token=HF_TOKEN)
print(f'Merged model uploaded to {CONFIG["hf_repo_merged"]}')

# Upload GGUF
create_repo(CONFIG['hf_repo_gguf'], exist_ok=True, token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=CONFIG['gguf_path'],
    path_in_repo=Path(CONFIG['gguf_path']).name,
    repo_id=CONFIG['hf_repo_gguf'],
    token=HF_TOKEN,
)
print(f'GGUF uploaded to {CONFIG["hf_repo_gguf"]}')

## 12. Smoke-test the model on Kaggle

Verify the merged model produces well-formed tool calls before we leave Kaggle.

In [ ]:
# ====================================================================
# Smoke-test: verify the fine-tuned model produces well-formed tool calls.
#
# M0-fix-22: previous version called `mod.generate(**inputs, ...)` where
# `inputs` was the result of apply_chat_template(..., tokenize=True, return_tensors='pt')
# which is a dict like {'input_ids': tensor, 'attention_mask': tensor}.
# Calling `.to(device)` on the dict (not the tensors) sometimes returns
# a non-mapping, breaking `**inputs`. Fix: move each tensor individually,
# and use Unsloth's FastLanguageModel.for_inference() for fast inference.
# ====================================================================
from transformers import AutoModelForCausalLM, AutoTokenizer
from unsloth import FastLanguageModel
import torch, json

test_messages = [
    {'role': 'system', 'content': "You are Thursday, an offline AI that controls the user's PC via tool calls. Emit exactly ONE tool call per turn. Current OS: Linux (GNOME)."},
    {'role': 'user', 'content': 'Hey Thursday, set an alarm for 7 am tomorrow.'},
]
test_tools = json.load(open(REPO / 'data' / 'schema' / 'tools.json'))['tools']

tok = AutoTokenizer.from_pretrained(CONFIG['merged_dir'])
mod = AutoModelForCausalLM.from_pretrained(CONFIG['merged_dir'], torch_dtype=torch.float16, device_map='auto')

# Enable Unsloth's fast inference path (2x faster generation)
FastLanguageModel.for_inference(mod)

# Build inputs with chat template — returns BatchEncoding (dict-like) when return_tensors='pt'
inputs = tok.apply_chat_template(
    test_messages,
    tools=test_tools,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
)

# Move each tensor to the model's device individually (avoid .to() on dict)
device = mod.device if hasattr(mod, 'device') else next(mod.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}
input_len = inputs['input_ids'].shape[1]
print(f'Input length: {input_len} tokens')

# Generate — pass inputs as a dict (not **inputs, which sometimes breaks)
with torch.no_grad():
    out = mod.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs.get('attention_mask'),
        max_new_tokens=200,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
    )

# Decode only the newly-generated tokens
generated = out[0][input_len:]
decoded = tok.decode(generated, skip_special_tokens=False)
print('--- model output ---')
print(decoded)
print('--- end ---')

# Quick sanity check: did the model emit a tool_call structure?
if '<tool_call>' in decoded or 'tool_calls' in decoded or 'set_alarm' in decoded:
    print('PASS: model emitted a tool-call structure')
else:
    print('NOTE: model did not emit obvious tool-call structure. May need more training.')

## 13. Download the GGUF locally (instructions)

Once the GGUF is on HF Hub, install it on your potato PC:

```bash
# Option A: Ollama (recommended)
ollama pull hf.co/skyro777/thursday-ai-v0.1-gguf:Q4_K_M
ollama run hf.co/skyro777/thursday-ai-v0.1-gguf:Q4_K_M

# Option B: llama.cpp direct
wget https://huggingface.co/skyro777/thursday-ai-v0.1-gguf/resolve/main/thursday-ai-v0.1-Q4_K_M.gguf
./llama-server --model thursday-ai-v0.1-Q4_K_M.gguf --threads 4 --ctx-size 4096 --port 8080
```

On an i5-3rd-gen (AVX only, no AVX2), build llama.cpp with:

```bash
cmake -B build -DLLAMA_AVX2=OFF -DLLAMA_AVX=ON -DLLAMA_FMA=ON -DLLAMA_F16C=ON
cmake --build build --config Release
```

Expected throughput: ~6-10 tok/s for short prompts. Voice-reply latency target: < 1.5 s.

---

**Notebook ends here.** Next steps after a successful run:

1. Run `eval/thursday_eval_v1.json` against the GGUF (M4)
2. Wire the runtime orchestrator (M3)
3. Iterate on dataset weaknesses (M2 → full 90k)